# Examine the Euler condition-aware training data

Interactive EDA for the **80 Euler vehicles** that train the condition-aware SoH forecaster (the `data/euler/features/feature_table.parquet` cohort). Covers: cohort overview, the SoH trajectories the model learns from, a per-vehicle **data-quality scorecard**, the garbage in `batteryRemainingCapacity` (the channel SoH is built on), feature completeness, and a drill-down into any single vehicle.

> The quality cell reads all 80 dense parquets — it takes ~1–2 minutes.

In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')
import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
TEAL,AMBER,RED,GREY='#1f9e8f','#e0922b','#d4504e','#9fb3c8'
EFT=pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin','month'])
VINS=sorted(EFT.vin.unique())
print('training cohort:', EFT.vin.nunique(), 'vehicles,', len(EFT), 'vin-months,', EFT.shape[1], 'columns')

## 1. Cohort overview — who's in the training set

In [ ]:
yr=lambda v:('20'+re.match(r'MD9EM.DL(\d{2})',v).group(1)) if re.match(r'MD9EM.DL(\d{2})',v) else '?'
g=EFT.groupby('vin')
ov=pd.DataFrame({'months':g.size(),'soh_start':g['soh'].first(),'soh_end':g['soh'].last(),'odo_end':g['odo_max'].last()})
ov['drop']=ov.soh_start-ov.soh_end; ov['year']=[yr(v) for v in ov.index]
ov['kind']=np.where(ov['drop']>=2,'degrader','flat')
print('by manufacture year:', ov.year.value_counts().sort_index().to_dict())
print('degraders (drop>=2pp):', int((ov.kind=='degrader').sum()), '| flat:', int((ov.kind=='flat').sum()))
print('history months: median %d (min %d, max %d)'%(ov.months.median(),ov.months.min(),ov.months.max()))
fig,ax=plt.subplots(1,2,figsize=(11,3.4))
ax[0].hist(ov.months,bins=20,color=TEAL); ax[0].set_title('history length'); ax[0].set_xlabel('months/vehicle')
ax[1].hist(ov['drop'],bins=20,color=AMBER); ax[1].axvline(2,color=RED,ls='--')
ax[1].set_title('observed SoH drop (flat < 2pp)'); ax[1].set_xlabel('pp'); plt.tight_layout()

## 2. SoH trajectories — the training target

Each line is one vehicle's calculated BMS-capacity SoH. The model learns decline mostly from the **degraders** (red); flat vehicles (grey) carry little signal.

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
for vin,gg in EFT.groupby('vin'):
    deg=(gg.soh.iloc[0]-gg.soh.iloc[-1])>=2
    ax.plot(gg.age_months,gg.soh,color=(RED if deg else GREY),alpha=0.55,lw=1)
ax.axhline(80,color=AMBER,ls='--',label='80% EoFL'); ax.set_ylim(55,101)
ax.set_xlabel('age (months)'); ax.set_ylabel('SoH %'); ax.legend()
ax.set_title('SoH trajectories of the 80 training vehicles (red = degrader, grey = flat)'); plt.tight_layout()

## 3. Per-vehicle data-quality scorecard

Reads the raw dense parquets and scores each vehicle on history, high-SoC support feeding the SoH, within-month capacity noise, `batteryRemainingCapacity` garbage rate, and feature fill. **Slow cell.**

In [ ]:
BND={'batterySoc':(0,100),'batteryCurrent':(-200,200),'batteryVoltage':(40,120),
     'batteryRemainingCapacity':(0,200),'batteryTemperature':(-20,80),'cellImbalance':(0,5000)}
rows=[]; OOB={c:0 for c in BND}; TOT=0
for vin in VINS:
    gg=EFT[EFT.vin==vin]; fp=Path(f'data/euler/dense/{vin}.parquet')
    rec=dict(vin=vin[-9:],months=len(gg),drop=round(gg.soh.iloc[0]-gg.soh.iloc[-1],1),
             imb_fill=round(gg['imbalance_mean'].notna().mean()*100) if 'imbalance_mean' in gg else 0)
    if fp.exists():
        d=pd.read_parquet(fp); TOT+=len(d)
        t=pd.to_datetime(d['t']) if 't' in d else pd.to_datetime(d['eventAt'].astype('int64'),unit='ms')
        bad=pd.Series(False,index=d.index)
        for c,(lo,hi) in BND.items():
            if c in d:
                s=pd.to_numeric(d[c],errors='coerce'); oob=s.notna()&~s.between(lo,hi)
                OOB[c]+=int(oob.sum()); bad|=oob; d[c]=s.where(s.between(lo,hi))
        rec['remcap_garbage%']=round(pd.to_numeric(pd.read_parquet(fp,columns=['batteryRemainingCapacity']).iloc[:,0],errors='coerce').pipe(lambda s:(s.notna()&~s.between(0,200)).mean())*100,1)
        hi=d[(d.batterySoc.between(95,100))&(d.batteryRemainingCapacity>0)].copy()
        rec['hi_per_mo']=round(hi.groupby(t[hi.index].dt.to_period('M')).size().median()) if len(hi)>10 else 0
        if len(hi)>10:
            hi['fc']=hi.batteryRemainingCapacity/(hi.batterySoc/100); hi['m']=t[hi.index].dt.to_period('M')
            cv=hi.groupby('m')['fc'].apply(lambda x:(x.std()/x.median()) if len(x)>3 and x.median()>0 else np.nan)
            rec['cap_noise%']=round(float(np.nanmedian(cv))*100,1)
    rows.append(rec)
Q=pd.DataFrame(rows)
def tier(r):
    if r.months<8 or r.get('hi_per_mo',0)<15 or r.get('cap_noise%',0)>12 or r.get('remcap_garbage%',0)>5: return 'LOW'
    if r.months>=12 and r.get('hi_per_mo',0)>=40 and r.get('cap_noise%',9)<6 and r.get('remcap_garbage%',9)<1: return 'HIGH'
    return 'MED'
Q['quality']=Q.apply(tier,axis=1)
print('quality tiers:', Q.quality.value_counts().reindex(['HIGH','MED','LOW']).fillna(0).astype(int).to_dict())
Q.sort_values(['quality','remcap_garbage%'],ascending=[True,False]).head(12)

## 4. The dirty channel — `batteryRemainingCapacity`

SoH is built on BMS remaining-capacity, which is by far the noisiest signal (scale-error/sentinel values like 1117 or 155540 Ah for a ~130 Ah pack). The robust pipeline clips it; 101 *other* vehicles were dropped because theirs was unrecoverable.

In [ ]:
ob=pd.Series(OOB)/TOT*100
fig,ax=plt.subplots(figsize=(8,3.2))
ob.sort_values().plot.barh(ax=ax,color=[RED if c=='batteryRemainingCapacity' else GREY for c in ob.sort_values().index])
ax.set_xlabel('% of raw rows out of physical bounds'); ax.set_title('Garbage rate by signal'); plt.tight_layout()
print('remaining-capacity garbage: %.2f%% of all rows'%ob['batteryRemainingCapacity'])

## 5. Feature completeness & distributions

Which model features are well-populated, and what their values look like across the cohort.

In [ ]:
FEATS=['ah_throughput','cur_abs_p95','volt_mean','temp_mean','temp_max','dod_mean','frac_soc_high',
       'frac_soc_low','imbalance_mean','km_month','odo_max','soc_mean']
fill=EFT[FEATS].notna().mean().sort_values()*100
fig,ax=plt.subplots(1,2,figsize=(12,4))
fill.plot.barh(ax=ax[0],color=TEAL); ax[0].set_xlabel('% non-null'); ax[0].set_title('feature completeness'); ax[0].set_xlim(0,100)
for f,c in [('temp_mean',TEAL),('dod_mean',AMBER),('imbalance_mean',RED)]:
    ax[1].hist(EFT[f].dropna(),bins=30,alpha=0.5,label=f)
ax[1].legend(); ax[1].set_title('selected feature distributions'); plt.tight_layout()

## 6. Drill into one vehicle

`examine(vin)` shows a vehicle's calculated SoH and its raw high-SoC capacity (with the garbage that gets clipped). Defaults to the clearest degrader; pass any VIN tail (e.g. `examine('217372')`).

In [ ]:
def examine(vin_tail=None):
    if vin_tail is None:
        ovd=ov[ov.months>=12]; vin=ovd['drop'].idxmax()
    else:
        vin=[v for v in VINS if v.endswith(vin_tail)][0]
    gg=EFT[EFT.vin==vin].sort_values('month'); d=pd.read_parquet(f'data/euler/dense/{vin}.parquet')
    t=pd.to_datetime(d['t']) if 't' in d else pd.to_datetime(d['eventAt'].astype('int64'),unit='ms')
    soc=pd.to_numeric(d['batterySoc'],errors='coerce'); rc=pd.to_numeric(d['batteryRemainingCapacity'],errors='coerce')
    hi=(soc.between(95,100))&(rc>0); fc=(rc/(soc/100)).where(hi)
    fig,ax=plt.subplots(1,2,figsize=(13,4))
    ax[0].plot(gg.age_months,gg.soh,'o-',color=TEAL); ax[0].axhline(80,color=AMBER,ls='--')
    ax[0].set_title(f'{vin[-9:]} — calculated SoH'); ax[0].set_xlabel('age (months)'); ax[0].set_ylabel('SoH %')
    ax[1].scatter(t,fc,s=3,alpha=0.3,color=TEAL,label='high-SoC full_cap')
    ax[1].axhspan(0,200,alpha=0.05,color='green'); ax[1].set_ylim(0,260)
    g_in=fc.between(0,200).mean(); g_all=(rc.notna()&~rc.between(0,200)).mean()*100
    ax[1].set_title(f'{vin[-9:]} — raw remaining-capacity (garbage {g_all:.1f}%)'); ax[1].set_ylabel('Ah'); ax[1].legend()
    plt.tight_layout(); return vin
examine()

### Takeaways
- The **SoH target is clean** after the robust pipeline (low within-month capacity noise, abundant high-SoC support), but the foundational `batteryRemainingCapacity` channel is the **dirtiest signal**.
- The decline signal rests on the **~54 degraders**; the ~26 flat vehicles add little.
- `imbalance_mean` is only partly populated (older vehicles) — the model tolerates the NaNs.
- Per-vehicle scorecard is also saved at `data/manifests/euler_data_quality.csv`.